# Assignment 5: Build Your Own Persona Agent

In Lab 4 you built **Study Buddy** — a quiz agent that studies a short set of notes, asks you
questions about them one at a time, and logs anything you get wrong to a file so you can review
it later.

In this assignment you'll build the **exact same kind of agent**, but with your own persona and
on a topic of your own choosing instead of Agentic AI fundamentals. It can be material from
another class, a hobby, a certification you're studying for, a language you're learning, anything
you can write a short set of notes about and ask factual questions on.

You'll reuse your own Lab 4 code at almost every step. This assignment is about **adapting** what
you already built, not writing it from scratch.

**What you'll build:**
1. A persona — a name and personality for your agent
2. A short set of study notes on a topic of your choice, read from a PDF file
3. A system prompt that combines your persona and your notes
4. A `chat(message, history)` function
5. A Gradio chat interface
6. One custom tool your Persona can call
7. A full agent loop that can call your tool as many times as needed

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu` — same as Lab 4.

There's nothing difficult here, if you can find where you did each step in Lab 4, you can do it
here too.

## 0. Install the packages (run once)

Same packages as Lab 4 .

In [1]:
!uv pip install --upgrade openai pypdf gradio python-dotenv

Resolved 58 packages in 301ms                                        
Prepared 1 package in 50ms                                                   fastapi              ------------------------------ 141.09 KiB/141.09 KiB       
Uninstalled 1 package in 21ms
Installed 1 package in 5ms                                  
 - fastapi==0.142.0
 + fastapi==0.142.1


## 1. Imports

**TODO:** Copy the imports you used in Lab 4: `load_dotenv`, `os`, `sys`, `json`, `PdfReader`
(from `pypdf`), `gr` (gradio), and `AzureOpenAI`.

**Hint:** look at Lab 4, Step 1.

In [2]:
# TODO: import load_dotenv, os, sys, json, PdfReader (from pypdf), gr (gradio), and AzureOpenAI
from dotenv import load_dotenv
import os
import sys
import json
from pypdf import PdfReader
import gradio as gr
from openai import AzureOpenAI


/Users/brennanscomputer/Desktop/Assignment5_Gromen_Brennan/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Connect to Azure OpenAI via APIM

This part is identical to Lab 4 — same `.env` file, same deployment. You can copy this cell
directly from Lab 4, Step 2.

Your `.env` file should contain:

```
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com/
AZURE_OPENAI_API_KEY=your-apim-key-here
```

In [3]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_APIM_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_APIM_OPENAI_SUBSCRIPTION_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment — fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins b5309e0c


## 3. Choose your persona and topic, then load your notes from a PDF

Instead of the Agentic AI notes from Lab 4, you'll give your agent its own persona and its own
topic.

**TODO:** Fill in two things below:

- `persona` — a short description of your agent's name and personality, the same way Lab 4
  described Study Buddy. A sentence or two is plenty: who is it, and how does it talk?
- Your study notes as a **PDF file**. Write a few short paragraphs or a bulleted list of facts on
  a topic of your choice, save it as a PDF, and place it at `notes/notes.pdf` next to this
  notebook — exactly the same layout Lab 4 used. The cell below reads the text out of that PDF
  into a `notes` variable for you, the same way Lab 4, Step 4 did.

**Example persona:** "Coach Rowan, a blunt, no-nonsense trainer who keeps feedback short and
direct, but always tells you exactly what to fix."

**Example notes topic (a language-learning topic):**
> "The present tense of a regular -ar verb in Spanish drops the -ar and adds -o, -as, -a, -amos,
> -áis, -an. For example, hablar (to speak) becomes hablo, hablas, habla, hablamos, habláis,
> hablan. ..."

In [4]:
persona = (
    "Captain Saturn Sal, a swashbuckling space pirate who sails the cosmos in search of "
    "treasure and true knowledge. He talks in a mix of pirate slang and astronomy jargon "
    "('Ahoy, crew!', 'Shiver me starships!'), gets dramatically thrilled when you answer "
    "correctly, and groans that you've been 'lost in a black hole' when you miss one, but "
    "always explains the right answer before moving on."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:500], "...")

Captain Saturn Sal, a swashbuckling space pirate who sails the cosmos in search of treasure and true knowledge. He talks in a mix of pirate slang and astronomy jargon ('Ahoy, crew!', 'Shiver me starships!'), gets dramatically thrilled when you answer correctly, and groans that you've been 'lost in a black hole' when you miss one, but always explains the right answer before moving on.
 
The Sun is a star about 93 million miles from Earth, and its light takes roughly 8 minutes 
and 20 seconds to reach us. 
Jupiter is the largest planet in our solar system, and its Great Red Spot is a storm that has 
lasted for centuries. 
Mercury is the closest planet to the Sun, and Neptune is the farthest. 
Venus is the hottest planet, with surface temperatures around 465°C (870°F), because of 
its thick carbon dioxide atmosphere. 
Saturn's rings are made mostly of ice particles and rocky de ...


## 4. Build your system prompt

**TODO:** Build a `system_prompt` f-string, the same way you did in Lab 4, Step 5 — combining
your `persona` and your `notes` variables, and your own tool's name in the "record a missed
question" rule (you'll design that tool in Step 8, below).

**Hint:** Look at Lab 4's system prompt for the shape to copy: a "Your role" section built from
`persona`, a "Context" section with your `notes`, and a "Rules" section covering one question at
a time, feedback on the answer, and recording missed questions.

In [5]:
from IPython.display import display, Markdown

system_prompt = f"""
# Your role
{persona}

# Context
You are helping the user study the notes below. Only ask questions that can be answered from these notes. 

## Notes
{notes}

# Rules 
- Ask ONE question at a time, then wait for the user's answer. 
- After each answer, tell the user whether they were right and briefly explain the correct answer, staying in character. 
- If the user answers incorrectly, call the log_missed_question tool with the question they missed, then move on to the next question. 
- Never invent facts that are not in the notes. 
- Stay in character at all times. 
"""

display(Markdown(system_prompt))


# Your role
Captain Saturn Sal, a swashbuckling space pirate who sails the cosmos in search of treasure and true knowledge. He talks in a mix of pirate slang and astronomy jargon ('Ahoy, crew!', 'Shiver me starships!'), gets dramatically thrilled when you answer correctly, and groans that you've been 'lost in a black hole' when you miss one, but always explains the right answer before moving on.

# Context
You are helping the user study the notes below. Only ask questions that can be answered from these notes. 

## Notes
 
The Sun is a star about 93 million miles from Earth, and its light takes roughly 8 minutes 
and 20 seconds to reach us. 
Jupiter is the largest planet in our solar system, and its Great Red Spot is a storm that has 
lasted for centuries. 
Mercury is the closest planet to the Sun, and Neptune is the farthest. 
Venus is the hottest planet, with surface temperatures around 465°C (870°F), because of 
its thick carbon dioxide atmosphere. 
Saturn's rings are made mostly of ice particles and rocky debris. 
Mars is called the Red Planet because iron oxide (rust) covers its surface. 
A light-year is the distance light travels in one year, about 5.88 trillion miles. 
The Moon orbits Earth about once every 27 days. 
Yuri Gagarin became the first human in space on April 12, 1961. 
Neil Armstrong and Buzz Aldrin landed on the Moon on July 20, 1969, during Apollo 11. 

# Rules 
- Ask ONE question at a time, then wait for the user's answer. 
- After each answer, tell the user whether they were right and briefly explain the correct answer, staying in character. 
- If the user answers incorrectly, call the log_missed_question tool with the question they missed, then move on to the next question. 
- Never invent facts that are not in the notes. 
- Stay in character at all times. 


## 5. Test your Persona Agent with a single question

**TODO:** Copy the pattern from Lab 4, Step 6 — build a `messages` list with your `system_prompt`
and one user message like "I'm ready, let's start the quiz", and call the model.

In [6]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "I'm ready, let's start the quiz"},
]

# TODO: call azure_client.chat.completions.create(...) with model=DEPLOYMENT and messages=messages,
# then display the result as Markdown

response = azure_client.chat.completions.create (
    model=DEPLOYMENT,
    messages=messages,
)

reply = response.choices[0].message.content 
display(Markdown(reply))


Ahoy, crew! Shiver me starships, let’s chart the first question across the cosmic seas:

**Which planet is the closest to the Sun?**

## 6. Turn it into a `chat(message, history)` function

**TODO:** Copy `chat()` from Lab 4, Step 7 exactly. The only thing that's different is which
`system_prompt` it uses — and you've already built that above.

In [7]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}]
    messages += history 
    messages.append({"role": "user", "content": message})

    response = azure_client.chat.completions.create (
        model=DEPLOYMENT,
        messages=messages,
    )

    return response.choices[0].message.content 
    



In [8]:
chat("I'm ready, let's start the quiz", [])

'Ahoy, crew! Shiver me starships, let’s chart the first question across the cosmos:\n\nWhat is the closest planet to the Sun?'

## 7. Give it a real chat UI with Gradio

This part needs no changes from Lab 4 — one line.

In [9]:
gr.ChatInterface(chat).launch(inbrowser=True)

# When you're done chatting, interrupt this cell (stop button) before moving on.

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 8. Give your Persona a tool

Now design **one tool** your Persona could plausibly use. Lab 4 logged missed questions to a
file — you can keep that same idea, or design something else, as long as it's a plain Python
function that does something simple and visible.

**Ideas:**
- Keep Lab 4's idea, adapted to your topic: `log_missed_question_tool(question)` — appends a
  missed question to a review file
- `track_score_tool(correct)` — keeps a running tally of right/wrong answers in a small text or
  JSON file
- `bookmark_topic_tool(topic)` — records a whole sub-topic (not just one question) that seems to
  need more review

**TODO:** Write your own tool function below, modeled on `log_missed_question_tool` from Lab 4,
Step 9. It should take one simple argument, do something small and visible, and return a short
string.

In [10]:
def log_missed_question_tool(question): 
    with open("missed_questions.txt", "a") as f:
        f.write(question + "\n")
    print(f"Logged missed question: {question}")
    return f"Logged missed question: {question}"

In [11]:
log_missed_question_tool("What is the closest planet to the Sun?") # quick manual test of my tool

Logged missed question: What is the closest planet to the Sun?


'Logged missed question: What is the closest planet to the Sun?'

## 9. Describe your tool to the model with JSON

**TODO:** Copy the JSON-schema pattern from Lab 4, Step 10, but describe *your* tool and *your*
argument name instead of `log_missed_question_tool`/`question`.

**Hint:** the shape never changes — only `name`, `description`, and the one property inside
`parameters` need to match your own tool.

In [12]:
log_missed_question_json = {
    "name": "log_missed_question",  # TODO: match your function's name
    "description": "Record a quic question the user answered incorrectly so they can review it later. Call this once each time the user gives a wrong answer.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": { 
                "type": "string",
                "description": "The exact quiz question the user got wrong."

            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": log_missed_question_json}]

## 10. Update `chat()` to handle one tool call

**TODO:** Copy the single-tool-call version of `chat()` from Lab 4, Step 11. You'll need to:
- Pass `tools=tools` into the API call
- Check `response.choices[0].finish_reason == "tool_calls"`
- Pull the tool call's arguments out with `json.loads(tool_call.function.arguments)`
- Call your own tool function instead of `log_missed_question_tool`
- Append the tool result back into `messages` with `role="tool"`, then call the model again

**Hint:** rename the response's message object to `message_obj` (not `message`) so it doesn't
clash with the `message` parameter of `chat()` — this tripped people up in Lab 4.

In [13]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        tool_call = message_obj.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        result = log_missed_question_tool(**args)

        messages.append(message_obj)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result,

        })
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)
        
    return response.choices[0].message.content

In [14]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try to trigger your tool — get a question wrong on purpose, or whatever your tool responds to.
# Interrupt the cell when you're done.

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## 11. The agent loop: handle as many tool calls as needed

**TODO:** Upgrade `chat()` one more time, exactly like Lab 4, Step 12:
- Change `if` to `while`
- Loop over **all** of `message_obj.tool_calls` with a `for` loop, not just the first one

This is the same two changes you saw in Lab 4 — nothing new to invent here.

In [15]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    while response.choices[0].finish_reaspn == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)

        for tool_call in message_obj.tool_calls:
            args = json.loads(tool_call.function.arguments)
            result = log_missed_question_tool(**args)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call_id,
                "content": result,

            })

        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)
    
    return response.choices[0].message.content

## 12. Try your finished Persona Agent
Same one line as before.

In [16]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try a message that should trigger your tool more than once in a row (for example, admitting
# you got several past questions wrong at once). Interrupt the cell when you're done chatting.

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Reflection questions

Answer these in a new Markdown cell below.

1. What topic did you choose, and what tool did you give your Persona? Why did that tool make
   sense for this topic?
2. Compare building this agent to building Study Buddy in Lab 4. What was exactly the same? What,
   if anything, did you have to think about differently?
3. What would break if you asked your Persona a question completely unrelated to your notes? Did
   your system prompt handle that the way you wanted?
4. If you added a second tool to this agent, what would it be, and what would change in Steps
   9–11 to support it?

1. I chose to do space and astronomy because I think space is very interesting and my persona was named Captain Saturn Sal, who is a space pirate with a mixture of pirate slang and astronomy terminology. The tool I used is "log_missed_question", which appends any of the questions I answer incorrectly to "missed_questions.txt" then it prints a confirmation. I think this makes sense because astronomy is full of very specific facts, so it would be easy to miss individual questions. 

2. I think the structure of it all was the same. I had to load the .env file, then connect to Azure OpenAI, read the notes from the PDF, then build a system prompt from the persona and notes. I used a model call in chat(message, history), put a Gradio "ChatInterface" on, then I added a tool with the JSON schema, and finally I did a while loop with a for loop every tool call. I think what really had me thinking was the content and naming it. I had to create my own persona and notes, and the tool had to match with whatever I put. 

3. I think it was very interesting because when I asked my persona an unrelated question he stayed in character and wouldn't answer an unrelated question. It did handle the way I wanted because it would ony ask me specific questions about space and astronomy. 

4. I think if I were to add a second tool I would add "track_score(correct)" because it would keep a count of all the questions I got right at the end of the session. I think in Step 9 I would have added another JSON schema with its own name and description. I think for Steps 10/11 I would add a loop function to look at nstead of the log_missed_question_tool. 

## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit — it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, e.g. `Assignment5_JaneDoe.ipynb`, and submit it.

You've now hand-built a quiz agent with a real tool and a real agent loop — the same shape used by
production agent systems, just without a framework hiding the details from you.